##### MY470 Computer Programming

### Problem Set 3, AT 2025

#### \*\*\* Due 12:00 noon on Monday, November 10 \*\*\*

---
### Simulating contagion on a network

In this problem set, you are asked to write a program that simulates the contagion of disease or information on an empirical network. In academic research, contagion models have been used to study the properties of different types of networks. In practice, contagion models are extremely valuable to predict the spread of contagious disease such as the flu or STDs.

We will use the simplest of contagion models — the SI model. SI stands for "susceptible and infected". The SI model assumes that once a susceptible individual is infected, there is no recovery. This is a good representation for the spread of non-curable but non-deadly infectious disease such as Herpes simplex or for the spread of new technologies and knowledge.

In the SI model we will implement, we will start with a population where everyone is susceptible. Then we will randomly pick a small number of individuals ("Patients 0") and infect them. In the next period, all the contacts of the infected individuals will get infected (thus, we will assume that the probability of transmission is 1). And so on. We will repeat the process until everyone in the network is infected or until a certain number of periods have passed (the latter is necessary for networks that are not connected and have separate components; in such situations, it is possible that the contagion never reaches some individuals). 

We will run the model on a real network. For simplicity, we will reuse the co-authorship network we analyzed in Problem Set 1. So think about the contagion in this case as learning about a new research technique, empirical finding, or theoretical result.

#### Warning
Please always follow the specifications exactly, or you will lose points. Please also always use docstrings to describe your methods. We will subtract points from your mark if you do not use appropriate description of your code. Please do not import anything other than the random library in Problem 4.


---
### Problem 1: Working in a team

Work with your assigned partner to complete and submit the problem set. You can meet in person to discuss the division of labor but we expect you to use GitHub to communicate when coding your part and merging your contributions. We will  review the Issues, Pull request, and Wiki stats for your repository. You will get the full points for this problem if we find sufficient evidence that you have made use of GitHub as a collaboration tool. 

#### Hints

One reasonable way to divide the work is to assign Problem 2 to Student A and Problem 3 to Student B.


---
### Problem 2: Class for network

Create a class called `UndirectedNetwork`. The class should have the following instance attributes:

* `nodes` — a dictionary where the node id is a key and the value is a list with the ids of the node's neighbors (coauthors for our data); initially empty

The class should have the following instance methods:

* `add_node()` — takes `node_id` and initializes it as a key in `nodes` if it is not already there
* `add_neighbors()` — takes two arguments: `ego_id` and `alter_id` and adds `alter_id` to `ego_id`'s list of neighbors and `ego_id` to `alter_id`'s list of neighbors, if they are not already there
* `get_node_ids()` — a generator method that gives the ids of the nodes in the network
* `get_node_neighbors()` — a generator method that takes `node_id` and gives its neighbors
* `get_data()` – takes a relative path to a data file as a string parameter `fpath`. The file is assumed to be a tab-delimited edgelist, where lines with metadata begin with \# (similarly to the data file from Problem 1). The method reads the data and saves it in the `nodes` attribute of the instance.

The class should also have this magic method:

* Calling the `print()` function on an `UndirectedNetwork` object should print the number of nodes in the network. Please ensure it prints exactly like this "Undirected network with X nodes.", with X being a placeholder for the number of nodes.

In [1]:
# TODO: Write your class and its methods here


class UndirectedNetwork(object):
    """A class that represents an Undirected Network of nodes and their connections."""

    #create a dictionary
    def __init__(self): 
        """Creates an empty dictionary of nodes"""
        self.nodes = {}  

    #update dictionary with keys
    def add_node(self, node_ID): 
       """Takes a new nodeID and adds it as a key to the dictionary of nodes"""
       if node_ID not in self.nodes:
            self.nodes[node_ID] = []  

    #update values of the dictionary if missing
    def add_neighbors(self, ego_ID, alter_ID):
        """Takes two node IDs, ensures they exist as keys, and adds them as each other's values
        if not already there."""
        if alter_ID not in self.nodes:
            self.nodes[alter_ID] = []
        if ego_ID not in self.nodes:
            self.nodes[ego_ID] = []
        if alter_ID not in self.nodes[ego_ID]:
            self.nodes[ego_ID].append(alter_ID)
        if ego_ID not in self.nodes[alter_ID]:
            self.nodes[alter_ID].append(ego_ID)   

    #return the keys of the dictionary
    def get_node_ids(self):    
        """Returns the nodeIDs of the nodes in the dictionary"""                     
        for x in self.nodes:
            yield x 

    #return a value of a key in the dictionary
    def get_node_neighbors(self, node_ID):
        """Takes a node_ID and returns its neighbors"""
        for x in self.nodes[node_ID]:
            yield x 

    def get_data(self, fpath):
        """Takes a string as a file path, reads a tab delimited edgelist, 
        skips lines that start with #, and updates the nodes dictionary."""
        with open(fpath, 'r') as data:
            for line in data:
                if line.startswith('#'):
                    continue
                else:
                    node_ID = line.strip().split('\t')
                    node1, node2 = node_ID
                    self.add_neighbors(node1, node2)

    #magic method to print number of nodes in network
    def __str__(self):
        """Returns a string representation of 
        the number of nodes in the network"""
        X = str(len(self.nodes))
        return "Undirected network with " + X + " nodes."


### Test case for Problem 2

Test your class below. 

* Instantiate your `UnidrectedNetwork` class to create an instance.
* Use the `get_data` method to pupulate it by passing a relative path to the `ca-GrQc.txt` file in the `data` repository (use the same relative path as in the previous problem sets).
* Call print on the instance of the class.

In [2]:
# TODO: Test get_data here and print the intance of the class

network = UndirectedNetwork()

fpath = "../data/ca-GrQc.txt"
network.get_data(fpath)

print(network)

Undirected network with 5242 nodes.


---
### Problem 3: Class for SI model


Create a class called `SIModel` that has the following instance attributes:

* `network` — an instance of type UndirectedNetwork taken at instantiation;
* `susceptible_nodes` — a list of ids for nodes that are not yet infected; initially includes all nodes from `network`;
* `infected_nodes` — a list of ids for nodes that are infected; initially empty;
* `num_infected` — an integer that keeps track of the number of infected nodes; initially `0`.

The class shoul have the following methods:

* `initialize()` — takes an integer `n` to randomly select `n` number of nodes and infect them; then prints the number of infected nodes;
* `update()` — iterates over the susceptible nodes in random order and infects those who have at least one infected neighbor; then prints the number of infected nodes. The process should be asynchronous, in the sense that a node immediately becomes infected and will then infect any susceptible neighbors who are yet to be iterated over;
* `run()` — takes an integer `num_iter` and repeats `update` until all nodes are infected or until `update` has been run `num_iter` times; `num_iter` has a default value of 100.

The class should also have this magic method:

* Calling the `print()` function on a `SIModel` object should just print `num_infected`, and nothing else.


#### Hints

In this problem you will need to use functions from the `random` module. You can read more about it [here](https://docs.python.org/3/library/random.html).

Make sure the methods update all the relevant data attributes when called.

In [3]:
# TODO: Write your class and its methods here

# Import the random module  
import random 

# Create class SIModel.

class SIModel(object):
    """Simulates the spread of an infection in an undirected network using the SI model."""

    def __init__(self, network):
        """Takes an UndirectedNetwork and initializes 
        the SI model with all nodes as susceptible."""
        self.network = network
        self.susceptible_nodes = list(network.get_node_ids())
        self.infected_nodes = []
        self.num_infected = 0 
    
    # initialise method 
    def initialize(self, n):
        """Takes an integer n and infects n number of nodes 
        in the network and prints the number of infected nodes."""
        self.infected_nodes = random.sample(self.susceptible_nodes, n)
        self.susceptible_nodes = [i for i in self.susceptible_nodes if i not in self.infected_nodes]
        self.num_infected = len(self.infected_nodes)
        print(self.num_infected)

    # update method 
    def update(self):
        """Updates the infection process by infecting susceptible nodes with 
        infected neighbours prints the number of infected nodes"""

        for i in random.sample(self.susceptible_nodes.copy(), len(self.susceptible_nodes)):
            if any(neigh in self.infected_nodes for neigh in self.network.nodes[i]):
                self.infected_nodes.append(i)
                self.susceptible_nodes.remove(i)
                self.num_infected = len(self.infected_nodes)
        print(f"Number of infected nodes: {self.num_infected}")


    # run method 
    def run(self, num_iter = 100):
        """Accepts an optional int for number of iterations (default 100) 
        and runs the SI simulation until all nodes are infected"""
        for _ in range(num_iter):
            if len(self.susceptible_nodes) == 0:
                break
            else:
                self.update()


    # Magic method to print the numer of infected
    def __str__(self):
        """Returns the number of infected nodes in the network"""
        return str(self.num_infected)


### Test case for Problem 3

Run `SIModel` using the network instance you created in Problem 3:

* You should initialize the simulation with 3 seeds (`n=3`) and update for 30 iterations (`num_iter=30`).


In [4]:
# TODO: write your code here

model = SIModel(network)

model.initialize(3)
model.run(30)

3
Number of infected nodes: 675
Number of infected nodes: 2891
Number of infected nodes: 3920
Number of infected nodes: 4123
Number of infected nodes: 4161
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163
Number of infected nodes: 4163


### Evaluation

| Function | Score | Comment |
|:--------|:-----|:-------|
| GitHub | 2/2 | Make sure to close inactive issues. |
| UndirectedNetwork | 4/5 | UndirectedNetwork should remove self loops. |
| SIModel | 4/6 | Use get_node_neighbors rather than accessing attributes directly. |
| Legibility | 2/2 | |
| Modularity | 2/2 | You could use print(self) to report number of infected in Q3. |
| Efficiency | 2/3 | There is no need to sample from a copy - sample generates a new list. |
| **Total** | **16/20** | Good work. |
